# imports

In [ ]:
def _mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return np.median(np.abs(x - med))

In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from neuralforecast import NeuralForecast
import optuna
from sklearn.model_selection import TimeSeriesSplit
import numpy as np
import pandas as pd
from pathlib import Path
import pathlib
import math
import holidays
import json
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import os
import random
import torch
import warnings
import xgboost as xgb
import torch
torch.set_float32_matmul_precision("medium")
from sklearn.metrics import root_mean_squared_error
from neuralforecast.losses.pytorch import MSE
import numpy as np
from scipy.stats import gaussian_kde, norm
import holidays
from sklearn.ensemble import RandomForestRegressor
import re
from mlforecast.lag_transforms import RollingMean
from mlforecast import MLForecast
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import rmse
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet

def _mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return np.median(np.abs(x - med))


def get_holiday_calendar(country):
    if country == "Germany":
        return holidays.Germany()
    elif country == "Ireland":
        return holidays.Ireland()
    elif country == "Portugal":
        return holidays.Portugal()
    else:
        return holidays.Germany()


def select_top_correlated_weather_lags(
    train_df,
    weather_cols,
    forecast_horizon,
    top_k_per_weather=6,
    max_weather_lag=None,
):
    """
    For each weather variable, create lags only in the safe range
    forecast_horizon .. max_weather_lag and keep only the top-k lags
    with highest absolute correlation to y.
    """
    if max_weather_lag is None:
        max_weather_lag = forecast_horizon * 2

    df = train_df.copy().sort_values(["unique_id", "ds"]).reset_index(drop=True)

    selected_weather_lag_features = []

    for col in weather_cols:
        corr_rows = []
        g = df.groupby("unique_id")[col]

        for lag in range(forecast_horizon, max_weather_lag + 1):
            lag_name = f"{col}_lag_{lag}"
            lag_values = g.shift(lag)

            tmp = pd.DataFrame({
                "y": df["y"].values,
                lag_name: lag_values.values
            }).dropna()

            if len(tmp) < 10:
                corr = 0.0
            else:
                corr = tmp["y"].corr(tmp[lag_name])
                if pd.isna(corr):
                    corr = 0.0

            corr_rows.append({
                "feature": lag_name,
                "abs_corr": abs(corr),
            })

        corr_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
        best_feats = corr_df.head(top_k_per_weather)["feature"].tolist()
        selected_weather_lag_features.extend(best_feats)

    return sorted(selected_weather_lag_features)

def _mad(x):
    x = np.asarray(x, dtype=float)
    med = np.median(x)
    return np.median(np.abs(x - med))

def build_extra_exog_features(
    history_df,
    weather_cols,
    country,
    selected_exog,
    future_df=None,
):
    """
    Build only the selected exogenous features.

    Parameters
    ----------
    history_df : pd.DataFrame
        Must contain ['unique_id', 'ds'] + weather_cols.
        This is the historical context available before prediction.
    weather_cols : list[str]
    country : str
    selected_exog : list[str]
        Exact exogenous feature names to build.
    future_df : pd.DataFrame or None
        If provided, features are built on history + future and only future rows are returned.
        This is necessary for lagged weather exog at prediction time.

    Returns
    -------
    out_df : pd.DataFrame
        Contains ['unique_id', 'ds'] + selected_exog
    """
    history_df = history_df.copy()
    history_df = history_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    if future_df is not None:
        future_df = future_df.copy()
        future_df = future_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

        full_df = pd.concat([history_df, future_df], ignore_index=True)
        full_df = full_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)
        future_keys = future_df[["unique_id", "ds"]].copy()
    else:
        full_df = history_df.copy()
        future_keys = None

    df = full_df.copy()

    # --------------------------------------------------
    # Calendar features
    # --------------------------------------------------
    needed = set(selected_exog)

    if any(f in needed for f in [
        "minute", "hour", "day_of_week", "day_of_year", "week", "month",
        "year", "is_weekend", "holiday",
        "minute_sin", "minute_cos", "hour_sin", "hour_cos",
        "dayofweek_sin", "dayofweek_cos",
        "dayofyear_sin", "dayofyear_cos",
        "week_sin", "week_cos",
        "month_sin", "month_cos"
    ]):
        df["minute"] = df["ds"].dt.minute
        df["hour"] = df["ds"].dt.hour
        df["day_of_week"] = df["ds"].dt.dayofweek
        df["day_of_year"] = df["ds"].dt.dayofyear
        df["week"] = df["ds"].dt.isocalendar().week.astype(int)
        df["month"] = df["ds"].dt.month
        df["year"] = df["ds"].dt.year
        df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

        holiday_calendar = get_holiday_calendar(country)
        df["holiday"] = df["ds"].dt.normalize().map(
            lambda x: 1 if x in holiday_calendar else 0
        )

        minute_period = 60
        hour_period = 24
        week_period = 7
        month_period = 12
        year_period = 365.25

        df["minute_sin"] = np.sin(2 * np.pi * df["minute"] / minute_period)
        df["minute_cos"] = np.cos(2 * np.pi * df["minute"] / minute_period)
        df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / hour_period)
        df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / hour_period)
        df["dayofweek_sin"] = np.sin(2 * np.pi * df["day_of_week"] / week_period)
        df["dayofweek_cos"] = np.cos(2 * np.pi * df["day_of_week"] / week_period)
        df["dayofyear_sin"] = np.sin(2 * np.pi * df["day_of_year"] / year_period)
        df["dayofyear_cos"] = np.cos(2 * np.pi * df["day_of_year"] / year_period)
        df["week_sin"] = np.sin(2 * np.pi * df["week"] / week_period)
        df["week_cos"] = np.cos(2 * np.pi * df["week"] / week_period)
        df["month_sin"] = np.sin(2 * np.pi * df["month"] / month_period)
        df["month_cos"] = np.cos(2 * np.pi * df["month"] / month_period)

    # --------------------------------------------------
    # Raw weather
    # --------------------------------------------------
    for col in weather_cols:
        if col in needed and col not in df.columns:
            raise ValueError(f"Missing raw weather column: {col}")

    # --------------------------------------------------
    # Weather lag features only if selected
    # --------------------------------------------------
    lag_pattern = re.compile(r"^(.+)_lag_(\d+)$")

    for feat in selected_exog:
        m = lag_pattern.fullmatch(feat)
        if m:
            base_col = m.group(1)
            lag = int(m.group(2))
            if base_col in weather_cols:
                df[feat] = df.groupby("unique_id")[base_col].shift(lag)

    # Keep only requested output columns
    out_cols = ["unique_id", "ds"] + selected_exog
    out_df = df[out_cols].copy()

    if future_keys is not None:
        out_df = future_keys.merge(out_df, on=["unique_id", "ds"], how="left")

    return out_df


def empirical_bayes_threshold_from_importance(
    importances,
    feature_names,
    alpha=0.20,
    transform="log1p",
    central_prop=0.80,
    random_state=42,
):
    """
    Empirical-Bayes-style thresholding of one fitted model's feature importances.

    Parameters
    ----------
    importances : array-like of shape (n_features,)
        Non-negative feature importances.
    feature_names : list-like
        Names aligned with importances.
    alpha : float, default=0.20
        Local FDR cutoff. Smaller -> stricter selection.
    transform : {"log1p", None}, default="log1p"
        Optional transform before density modeling.
    central_prop : float, default=0.80
        Middle fraction of the transformed distribution used to estimate null center/spread.
    random_state : int, default=42
        For tiny jitter to break ties safely.

    Returns
    -------
    results_df : pd.DataFrame
        Columns:
        - feature
        - importance_raw
        - importance_transformed
        - local_fdr
        - selected
    threshold_raw : float
        Minimum raw importance among selected features.
        If no features pass, returns +inf.
    """
    imp = np.asarray(importances, dtype=float)
    if np.any(imp < 0):
        raise ValueError("Importances must be non-negative.")

    names = np.asarray(feature_names)
    if len(names) != len(imp):
        raise ValueError("feature_names and importances must have the same length.")

    rng = np.random.default_rng(random_state)
    eps = 1e-12
    imp_j = imp + eps * rng.normal(size=len(imp))

    if transform == "log1p":
        z = np.log1p(np.maximum(imp_j, 0.0))
    elif transform is None:
        z = imp_j.copy()
    else:
        raise ValueError("transform must be 'log1p' or None")

    # Robust empirical null from the central bulk
    q_low = (1.0 - central_prop) / 2.0
    q_high = 1.0 - q_low
    lo, hi = np.quantile(z, [q_low, q_high])
    z_central = z[(z >= lo) & (z <= hi)]

    mu0 = np.median(z_central)
    sigma0 = 1.4826 * _mad(z_central)
    sigma0 = max(sigma0, 1e-6)

    # Mixture density estimate from all transformed importances
    if len(np.unique(z)) < 2:
        # pathological case: all importances identical
        f_z = np.ones_like(z)
    else:
        kde = gaussian_kde(z)
        f_z = kde.evaluate(z)

    # Null density from robust Gaussian empirical null
    f0_z = norm.pdf(z, loc=mu0, scale=sigma0)

    # Conservative estimate of pi0
    pi0 = np.mean(z <= mu0 + sigma0)
    pi0 = float(np.clip(pi0, 0.50, 0.99))

    local_fdr = np.clip(pi0 * f0_z / np.maximum(f_z, 1e-12), 0.0, 1.0)
    selected = local_fdr <= alpha

    results_df = pd.DataFrame({
        "feature": names,
        "importance_raw": imp,
        "importance_transformed": z,
        "local_fdr": local_fdr,
        "selected": selected,
    }).sort_values(
        by=["selected", "importance_raw", "local_fdr"],
        ascending=[False, False, True]
    ).reset_index(drop=True)

    if results_df["selected"].any():
        threshold_raw = results_df.loc[results_df["selected"], "importance_raw"].min()
    else:
        threshold_raw = np.inf

    return results_df, threshold_raw



def select_features_rf_empirical_bayes(
    train_df,
    weather_cols,
    country,
    forecast_horizon,
    alpha=0.20,
    rf_params=None,
    fallback_top_k=25,
    top_k_per_weather=6,
    max_weather_lag=None,
):
    """
    Strict feature selection using:
    - target lags from forecast_horizon .. 2*forecast_horizon
    - weather lags ONLY from forecast_horizon .. max_weather_lag
    - calendar features
    - holiday
    - Fourier/cyclical features

    Raw weather is NOT included.
    """
    if max_weather_lag is None:
        max_weather_lag = forecast_horizon * 2

    df = train_df.copy()
    df = df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    # IMPORTANT: raw weather is NOT included anymore
    feat_df = df[["unique_id", "ds", "y"]].copy()

    # --------------------------------------------------
    # 1) TARGET LAGS
    # --------------------------------------------------
    target_lags = list(range(forecast_horizon, forecast_horizon * 2 + 1))
    for lag in target_lags:
        feat_df[f"lag_{lag}"] = feat_df.groupby("unique_id")["y"].shift(lag)

    # --------------------------------------------------
    # 2) SAFE WEATHER LAGS ONLY
    # --------------------------------------------------
    best_weather_lag_features = select_top_correlated_weather_lags(
        train_df=train_df,
        weather_cols=weather_cols,
        forecast_horizon=forecast_horizon,
        top_k_per_weather=top_k_per_weather,
        max_weather_lag=max_weather_lag,
    )

    for feat in best_weather_lag_features:
        m = re.fullmatch(r"(.+)_lag_(\d+)", feat)
        if m:
            base_col = m.group(1)
            lag = int(m.group(2))
            feat_df[feat] = df.groupby("unique_id")[base_col].shift(lag)

    # --------------------------------------------------
    # 3) CALENDAR + HOLIDAYS
    # --------------------------------------------------
    feat_df["minute"] = df["ds"].dt.minute
    feat_df["hour"] = df["ds"].dt.hour
    feat_df["day_of_week"] = df["ds"].dt.dayofweek
    feat_df["day_of_year"] = df["ds"].dt.dayofyear
    feat_df["week"] = df["ds"].dt.isocalendar().week.astype(int)
    feat_df["month"] = df["ds"].dt.month
    feat_df["year"] = df["ds"].dt.year
    feat_df["is_weekend"] = (feat_df["day_of_week"] >= 5).astype(int)

    holiday_calendar = get_holiday_calendar(country)
    feat_df["holiday"] = df["ds"].dt.normalize().map(
        lambda x: 1 if x in holiday_calendar else 0
    )

    # --------------------------------------------------
    # 4) FOURIER / CYCLICAL FEATURES
    # --------------------------------------------------
    minute_period = 60
    hour_period = 24
    week_period = 7
    month_period = 12
    year_period = 365.25

    feat_df["minute_sin"] = np.sin(2 * np.pi * feat_df["minute"] / minute_period)
    feat_df["minute_cos"] = np.cos(2 * np.pi * feat_df["minute"] / minute_period)
    feat_df["hour_sin"] = np.sin(2 * np.pi * feat_df["hour"] / hour_period)
    feat_df["hour_cos"] = np.cos(2 * np.pi * feat_df["hour"] / hour_period)
    feat_df["dayofweek_sin"] = np.sin(2 * np.pi * feat_df["day_of_week"] / week_period)
    feat_df["dayofweek_cos"] = np.cos(2 * np.pi * feat_df["day_of_week"] / week_period)
    feat_df["dayofyear_sin"] = np.sin(2 * np.pi * feat_df["day_of_year"] / year_period)
    feat_df["dayofyear_cos"] = np.cos(2 * np.pi * feat_df["day_of_year"] / year_period)
    feat_df["week_sin"] = np.sin(2 * np.pi * feat_df["week"] / week_period)
    feat_df["week_cos"] = np.cos(2 * np.pi * feat_df["week"] / week_period)
    feat_df["month_sin"] = np.sin(2 * np.pi * feat_df["month"] / month_period)
    feat_df["month_cos"] = np.cos(2 * np.pi * feat_df["month"] / month_period)

    feat_df = feat_df.copy()  # optional defragmentation

    # --------------------------------------------------
    # 5) DROP NA
    # --------------------------------------------------
    feat_df = feat_df.dropna().reset_index(drop=True)

    X = feat_df.drop(columns=["unique_id", "ds", "y"])
    y = feat_df["y"].values

    if rf_params is None:
        rf_params = {
            "n_estimators": 500,
            "random_state": 42,
            "n_jobs": -1,
            "max_features": "sqrt",
        }

    rf = RandomForestRegressor(**rf_params)
    rf.fit(X, y)

    importances = rf.feature_importances_

    importance_df, threshold_raw = empirical_bayes_threshold_from_importance(
        importances=importances,
        feature_names=X.columns.tolist(),
        alpha=alpha,
        transform="log1p",
        central_prop=0.80,
        random_state=42,
    )

    selected_features = importance_df.loc[importance_df["selected"], "feature"].tolist()

    if len(selected_features) == 0:
        selected_features = (
            importance_df.sort_values("importance_raw", ascending=False)
            .head(min(fallback_top_k, len(importance_df)))["feature"]
            .tolist()
        )
        importance_df["selected"] = importance_df["feature"].isin(selected_features)
        threshold_raw = importance_df.loc[
            importance_df["feature"].isin(selected_features), "importance_raw"
        ].min()

    importance_df = importance_df.sort_values(
        ["selected", "importance_raw"],
        ascending=[False, False]
    ).reset_index(drop=True)

    print(f"RF empirical-Bayes threshold (raw importance): {threshold_raw:.8f}")
    print(f"Number of selected features: {len(selected_features)}")

    return selected_features, importance_df, feat_df


# ============================================================
# GLOBAL SETTINGS FOR LIGHTWEIGHT HPO
# ============================================================
MAX_STEPS = 500
VAL_CHECK_STEPS = MAX_STEPS // 10

from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
import pandas as pd
import numpy as np

def build_feature_candidates(freq="15min"):
    return MLForecast(
        models=[],
        freq=freq,
        lags=[1, 2, 3, 4, 8, 12, 24, 96, 97, 98, 99, 100, 192, 288, 672],
        lag_transforms={
            1: [RollingMean(window_size=4), RollingMean(window_size=8)],
            4: [RollingMean(window_size=4)],
            96: [RollingMean(window_size=4), RollingMean(window_size=8)],
        },
        date_features=["hour", "dayofweek", "month"],
    )

def make_train_features(train_df, weather_cols, freq="15min"):
    fcst_features = build_feature_candidates(freq=freq)

    features_df = fcst_features.preprocess(
        train_df,
        id_col="unique_id",
        time_col="ds",
        target_col="y",
        static_features=[]
    )

    # Keep only rows where lagged features are available
    features_df = features_df.dropna().reset_index(drop=True)

    # Candidate predictors = all except id/time/target
    feature_cols = [
        c for c in features_df.columns
        if c not in ["unique_id", "ds", "y"]
    ]

    X = features_df[feature_cols].copy()
    y = features_df["y"].copy()

    return features_df, X, y, feature_cols






def extract_recipe_from_selected_features(selected_features, weather_cols):
    selected_lags = set()
    selected_extra_exog = []

    for feat in selected_features:
        m = re.fullmatch(r"lag_(\d+)", feat.lower())
        if m:
            selected_lags.add(int(m.group(1)))
        else:
            selected_extra_exog.append(feat)

    if not selected_lags:
        selected_lags = {96}

    return {
        "lags": sorted(selected_lags),
        "lag_transforms": {},
        "date_features": [],
        "weather_features": [],
        "extra_exog_features": sorted(set(selected_extra_exog)),
    }

def split_train_val_test_global(df_nf, test_start, forecast_horizon, training_size, val_days=3, freq_minutes=15):
    """
    Split a long NeuralForecast dataframe into train / validation / test per series.

    Validation starts `val_days` before test_start and ends right before test_start.
    Test spans `forecast_horizon` steps starting at test_start.
    Training is the last `training_size` rows before validation starts.
    """
    test_start = pd.Timestamp(test_start)
    val_start = test_start - pd.Timedelta(days=val_days)
    test_end = test_start + pd.Timedelta(minutes=freq_minutes * forecast_horizon)

    train_parts = []
    val_parts = []
    test_parts = []

    expected_val_len = val_days * (24 * 60 // freq_minutes)

    for uid, g in df_nf.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)

        # train: everything before validation starts, keep only last training_size rows
        train_candidates = g[g["ds"] < val_start].copy()
        train_df_uid = train_candidates.iloc[-training_size:].copy()

        # validation: from val_start until just before test_start
        val_df_uid = g[(g["ds"] >= val_start) & (g["ds"] < test_start)].copy()

        # test: from test_start for forecast_horizon steps
        test_df_uid = g[(g["ds"] >= test_start) & (g["ds"] < test_end)].copy()

        # safety checks
        if len(train_df_uid) != training_size:
            raise ValueError(f"{uid}: expected {training_size} training rows, got {len(train_df_uid)}")

        if len(val_df_uid) != expected_val_len:
            raise ValueError(f"{uid}: expected {expected_val_len} validation rows, got {len(val_df_uid)}")

        if len(test_df_uid) != forecast_horizon:
            raise ValueError(f"{uid}: expected {forecast_horizon} test rows, got {len(test_df_uid)}")

        train_parts.append(train_df_uid)
        val_parts.append(val_df_uid)
        test_parts.append(test_df_uid)

    train_df = pd.concat(train_parts, ignore_index=True)
    val_df = pd.concat(val_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)

    return train_df, val_df, test_df


def build_global_nf_df(df_all, home_cols, weather_cols=None):
    """
    Convert a wide household load dataframe into NeuralForecast long format.

    Parameters
    ----------
    df_all : pd.DataFrame
        Index must be DatetimeIndex, columns include home_* and optional weather cols.
    home_cols : list
        List of household columns, e.g. ['home_1', 'home_2', ...]
    weather_cols : list or None
        Optional list of weather columns to attach to every home/timestamp row.

    Returns
    -------
    df_nf : pd.DataFrame
        Columns: unique_id, ds, y, [weather columns...]
    """
    if not isinstance(df_all.index, pd.DatetimeIndex):
        raise ValueError("df_all index must be a DatetimeIndex.")

    # keep time as a normal column
    df_base = df_all.reset_index().rename(columns={"timestamp": "ds"})

    # wide -> long for homes
    df_nf = df_base.melt(
        id_vars=["ds"],
        value_vars=home_cols,
        var_name="unique_id",
        value_name="y"
    )

    # add weather columns if requested
    if weather_cols is not None and len(weather_cols) > 0:
        weather_df = df_base[["ds"] + weather_cols].copy()
        df_nf = df_nf.merge(weather_df, on="ds", how="left")

    # sort for safety
    df_nf = df_nf.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return df_nf

def build_daily_profile_matrix(df_all, home_cols):
    df_tmp = df_all.copy()
    df_tmp["slot"] = df_tmp.index.hour * 4 + df_tmp.index.minute // 15

    profiles = []
    for home in home_cols:
        avg_profile = df_tmp.groupby("slot")[home].mean()
        avg_profile.name = home
        profiles.append(avg_profile)

    profile_df = pd.concat(profiles, axis=1).T
    profile_df.index.name = "home"

    return profile_df



def compute_average_rmse_per_cluster(val_df, val_preds_df, pred_col="ElasticNet"):
    val_compare_df = val_df.merge(val_preds_df, on=["unique_id", "ds"], how="left")

    rmse_rows = []
    for uid, g in val_compare_df.groupby("unique_id"):
        rmse_rows.append({
            "unique_id": uid,
            "RMSE": root_mean_squared_error(g["y"], g[pred_col])
        })

    rmse_per_home = pd.DataFrame(rmse_rows).sort_values("RMSE").reset_index(drop=True)
    avg_rmse_cluster = rmse_per_home["RMSE"].mean()

    return avg_rmse_cluster, rmse_per_home, val_compare_df

def rolling_forecasting_validation_predictions(
    train_df,
    val_df,
    h,
    model_params,
    selected_exog,
    weather_cols,
    country,
    freq="15min"
):
    rolling_train_df = train_df.copy()
    val_predictions = []

    val_starts = sorted(val_df["ds"].unique())[::h]

    for window_start in val_starts:
        model = make_pipeline(
            StandardScaler(),
            ElasticNet(
                alpha=model_params["alpha"],
                l1_ratio=model_params["l1_ratio"],
                fit_intercept=model_params["fit_intercept"],
                max_iter=10000,
                random_state=42,
            )
        )

        fcst = MLForecast(
            models={"ElasticNet": model},
            freq=freq,
            lags=model_params["lags"],
            lag_transforms=model_params["lag_transforms"],
            date_features=model_params["date_features"],
        )

        if len(selected_exog) > 0:
            rolling_train_exog = build_extra_exog_features(
                history_df=rolling_train_df[["unique_id", "ds"] + weather_cols].copy(),
                weather_cols=weather_cols,
                country=country,
                selected_exog=selected_exog,
                future_df=None,
            )

            rolling_train_df_fit = rolling_train_df[["unique_id", "ds", "y"]].merge(
                rolling_train_exog,
                on=["unique_id", "ds"],
                how="left"
            )
        else:
            rolling_train_df_fit = rolling_train_df[["unique_id", "ds", "y"]].copy()

        fcst.fit(
            rolling_train_df_fit,
            id_col="unique_id",
            time_col="ds",
            target_col="y",
            static_features=[]
        )

        future_chunk = val_df[
            (val_df["ds"] >= window_start) &
            (val_df["ds"] < window_start + pd.Timedelta(minutes=15 * h))
        ].copy()

        if len(selected_exog) > 0:
            future_exog = build_extra_exog_features(
                history_df=rolling_train_df[["unique_id", "ds"] + weather_cols].copy(),
                weather_cols=weather_cols,
                country=country,
                selected_exog=selected_exog,
                future_df=future_chunk[["unique_id", "ds"] + weather_cols].copy(),
            )

            X_df = future_exog.copy()

            raw_weather_in_X_df = [c for c in weather_cols if c in X_df.columns]
            print("Validation X_df raw weather columns:", raw_weather_in_X_df if raw_weather_in_X_df else "None")

            preds = fcst.predict(h=h, X_df=X_df)
        else:
            preds = fcst.predict(h=h)

        val_predictions.append(preds)

        rolling_train_df = pd.concat([rolling_train_df, future_chunk], ignore_index=True)
        rolling_train_df = rolling_train_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return pd.concat(val_predictions, ignore_index=True)


def objective(trial):

    model_params = {
        "alpha": trial.suggest_float("alpha", 1e-2, 10.0, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "lags": feature_recipe["lags"],
        "lag_transforms": feature_recipe["lag_transforms"],
        "date_features": feature_recipe["date_features"],
    }

    try:
        val_preds_df = rolling_forecasting_validation_predictions(
            train_df=train_df,
            val_df=val_df,
            h=forecast_horizon,
            model_params=model_params,
            selected_exog=sorted(
                set(feature_recipe["weather_features"] + feature_recipe["extra_exog_features"])
            ),
            weather_cols=weather_cols,
            country=country,
            freq="15min"
        )

        avg_rmse_cluster, _, _ = compute_average_rmse_per_cluster(
            val_df=val_df,
            val_preds_df=val_preds_df,
            pred_col="ElasticNet"
        )

        return avg_rmse_cluster

    except Exception as e:
        import traceback
        print(f"Trial failed: {e}")
        traceback.print_exc()
        return float("inf")

# start

In [ ]:

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

countries = ["Germany", "Ireland", "Portugal"]
#countries = ["Germany"]

days = ["day1", "day2", "day3", "day4", "day5"]
#days = ["day1"]


forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
feature_selection = True
plot_forecast = True
hyperparameter_opt = True
opt_trials = 20

weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

# -------------------------
# Read JSON with forecast days
# -------------------------
with open(days_json_path, "r") as f:
    dataset_days = json.load(f)

# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp")
    df_all = df_all.sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)

    # make the dataset of each country clustered
    df_nf = build_global_nf_df(df_all, home_cols, weather_cols)
    profile_df = build_daily_profile_matrix(df_all, home_cols)

    print(profile_df.head())
    print(profile_df.shape)   # should be (28, 96)

    scaler = StandardScaler()
    X_profile = scaler.fit_transform(profile_df)

    results = []
    for k in range(2, 6):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = kmeans.fit_predict(X_profile)
        score = silhouette_score(X_profile, labels)

        results.append({"k": k, "silhouette_score": score})

    results_df = pd.DataFrame(results).sort_values("silhouette_score", ascending=False)
    print(results_df)

    best_k = int(results_df.iloc[0]["k"])
    best_score = results_df.iloc[0]["silhouette_score"]

    print(f"Best k: {best_k}")
    print(f"Best silhouette score: {best_score:.4f}")

    best_kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    best_labels = best_kmeans.fit_predict(X_profile)

    cluster_profile_df = profile_df.copy()
    cluster_profile_df["cluster"] = best_labels

    print(cluster_profile_df["cluster"].sort_values())
    # finish clustering
    
    # -------------------------
    # Loop over days
    # -------------------------
    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")


        clusters=best_k
        cluster_test_preds_list = [] # for the predictions
        for cluster in range(clusters):
            print(cluster)

            # --------------------------------------------------
            # iterate over clusters
            # --------------------------------------------------
            selected_cluster = cluster   # change to 1 if you want the other one later

            cluster_homes = cluster_profile_df.index[cluster_profile_df["cluster"] == selected_cluster].tolist()

            print(f"Selected cluster: {selected_cluster}")
            print(f"Number of homes in cluster: {len(cluster_homes)}")
            print("Homes in cluster:")
            print(cluster_homes)


            # --------------------------------------------------
            # subset original dataframe to homes in this cluster
            # --------------------------------------------------
            cluster_cols = cluster_homes + weather_cols
            df_cluster_wide = df_all[cluster_cols].copy()

            print("\nCluster-wide dataframe head:")
            print(df_cluster_wide.head())


            df_cluster_nf = build_global_nf_df(
                df_all=df_cluster_wide,
                home_cols=cluster_homes,
                weather_cols=weather_cols
            )

            print("\nCluster long-format dataset:")
            print(df_cluster_nf.head(10))

            print("\nColumns:")
            print(df_cluster_nf.columns.tolist())

            print("\nShape:")
            print(df_cluster_nf.shape)

            print("\nUnique homes in long dataset:")
            print(df_cluster_nf['unique_id'].unique())

            train_df, val_df, test_df = split_train_val_test_global(
                df_nf=df_cluster_nf,
                test_start=date,
                forecast_horizon=forecast_horizon,
                training_size=training_size,
                val_days=3,
                freq_minutes=15
            )

            selected_features, importance_df, features_df = select_features_rf_empirical_bayes(
                train_df=train_df,
                weather_cols=weather_cols,
                country=country,
                forecast_horizon=forecast_horizon,
                alpha=0.20,
                rf_params={
                    "n_estimators": 500,
                    "random_state": 42,
                    "n_jobs": -1,
                    "max_features": "sqrt",
                },
                fallback_top_k=25,
                top_k_per_weather=6,
                max_weather_lag=forecast_horizon * 2,
            )


            print("All features:")
            print(features_df.columns.tolist())
            print("Top selected features:")
            print(selected_features)

            print("\nTop feature importances:")
            print(importance_df.head(20))


            feature_recipe = extract_recipe_from_selected_features(selected_features, weather_cols)
            selected_exog = feature_recipe["extra_exog_features"]

            print("\nFeature recipe:")
            print(feature_recipe)


            for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
                start = df["ds"].min()
                end = df["ds"].max()
                print(f"{name}: {start} to {end} (Shape: {df.shape})")

            study = optuna.create_study(direction="minimize")
            study.optimize(objective, n_trials=opt_trials, show_progress_bar=True)

            print("Best avg RMSE:", study.best_value)
            print("Best params:", study.best_params)

            best_params = study.best_params

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            train_val_df = train_val_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

            train_val_exog = build_extra_exog_features(
                history_df=train_val_df[["unique_id", "ds"] + weather_cols].copy(),
                weather_cols=weather_cols,
                country=country,
                selected_exog=selected_exog,
                future_df=None,
            )

            test_exog = build_extra_exog_features(
                history_df=train_val_df[["unique_id", "ds"] + weather_cols].copy(),
                weather_cols=weather_cols,
                country=country,
                selected_exog=selected_exog,
                future_df=test_df[["unique_id", "ds"] + weather_cols].copy(),
            )

            # Which exogenous features survived selection?
            selected_exog = sorted(set(feature_recipe["weather_features"] + feature_recipe["extra_exog_features"]))

            final_model = make_pipeline(
                StandardScaler(),
                ElasticNet(
                    alpha=best_params["alpha"],
                    l1_ratio=best_params["l1_ratio"],
                    fit_intercept=best_params["fit_intercept"],
                    max_iter=10000,
                    random_state=42,
                )
            )

            fcst_final = MLForecast(
                models={"ElasticNet": final_model},
                freq="15min",
                lags=feature_recipe["lags"],
                lag_transforms=feature_recipe["lag_transforms"],
                date_features=feature_recipe["date_features"],
            )

            if len(selected_exog) > 0:
                train_val_df_fit = train_val_df[["unique_id", "ds", "y"]].merge(
                    train_val_exog,
                    on=["unique_id", "ds"],
                    how="left"
                )

                test_df_fit = test_df[["unique_id", "ds"]].merge(
                    test_exog,
                    on=["unique_id", "ds"],
                    how="left"
                )
            else:
                train_val_df_fit = train_val_df[["unique_id", "ds", "y"]].copy()
                test_df_fit = None



            print("Train exog cols:", [c for c in train_val_df_fit.columns if c not in ["unique_id", "ds", "y"]][:20])
            print("Num train exog cols:", len([c for c in train_val_df_fit.columns if c not in ["unique_id", "ds", "y"]]))

            if test_df_fit is not None:
                print("Test exog cols:", [c for c in test_df_fit.columns if c not in ["unique_id", "ds"]][:20])
                print("Num test exog cols:", len([c for c in test_df_fit.columns if c not in ["unique_id", "ds"]]))

                train_exog_cols = set(train_val_df_fit.columns) - {"unique_id", "ds", "y"}
                test_exog_cols = set(test_df_fit.columns) - {"unique_id", "ds"}

                print("Same exog columns?", train_exog_cols == test_exog_cols)
                print("Missing in test:", sorted(train_exog_cols - test_exog_cols))
                print("Extra in test:", sorted(test_exog_cols - train_exog_cols))




            fcst_final.fit(
                train_val_df_fit,
                id_col="unique_id",
                time_col="ds",
                target_col="y",
                static_features=[]
            )

            if test_df_fit is not None:

                raw_weather_in_test_df_fit = [c for c in weather_cols if c in test_df_fit.columns]
                print("Final test X_df raw weather columns:", raw_weather_in_test_df_fit if raw_weather_in_test_df_fit else "None")
                
                test_preds_df = fcst_final.predict(h=forecast_horizon, X_df=test_df_fit)
            else:
                test_preds_df = fcst_final.predict(h=forecast_horizon)



            test_preds_wide = test_preds_df.pivot(
                index="ds",
                columns="unique_id",
                values="ElasticNet"
            ).sort_index()
            cluster_test_preds_list.append(test_preds_wide)
        final_test_preds_wide = pd.concat(cluster_test_preds_list, axis=1).sort_index()


        # --------------------------------------------------
        # save final combined test predictions
        # --------------------------------------------------
        save_dir = pathlib.Path(project_path) / "Outputs" / "Global models" / "ElasticNet"
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"prediction_ElasticNet_{day_name}_{country}.csv"

        final_test_preds_wide.to_csv(save_path, index=True)

        print(f"Saved final_test_preds_wide to: {save_path}")




####################################################################################################
COUNTRY: Germany
####################################################################################################
Detected 28 homes for Germany.
['home_1', 'home_2', 'home_3', 'home_4', 'home_5', 'home_6', 'home_7', 'home_8', 'home_9', 'home_10', 'home_11', 'home_12', 'home_13', 'home_14', 'home_15', 'home_16', 'home_17', 'home_18', 'home_19', 'home_20', 'home_21', 'home_22', 'home_23', 'home_24', 'home_25', 'home_26', 'home_27', 'home_28']
slot             0            1            2            3            4   \
home                                                                      
home_1   252.102996   261.048660   246.110175   257.327671   454.420217   
home_2   934.975708   902.009367   866.633501   901.735967   867.579963   
home_3  1066.707239  1058.940145   990.620198   993.305127   990.090731   
home_4   550.102459   525.683797   537.442987   514.869658   524.066859   

C:\Users\CR58XM\AppData\Local\Temp\ipykernel_16288\2011431153.py:381: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[feat] = feat_df.groupby("unique_id")[base_col].shift(lag)
C:\Users\CR58XM\AppData\Local\Temp\ipykernel_16288\2011431153.py:381: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feat_df[feat] = feat_df.groupby("unique_id")[base_col].shift(lag)
C:\Users\CR58XM\AppData\Local\Temp\ipykernel_16288\2011431153.py:381: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.in

RF empirical-Bayes threshold (raw importance): 0.00000000
Number of selected features: 49
All features:
['unique_id', 'ds', 'y', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation', 'direct_radiation', 'lag_96', 'lag_97', 'lag_98', 'lag_99', 'lag_100', 'lag_101', 'lag_102', 'lag_103', 'lag_104', 'lag_105', 'lag_106', 'lag_107', 'lag_108', 'lag_109', 'lag_110', 'lag_111', 'lag_112', 'lag_113', 'lag_114', 'lag_115', 'lag_116', 'lag_117', 'lag_118', 'lag_119', 'lag_120', 'lag_121', 'lag_122', 'lag_123', 'lag_124', 'lag_125', 'lag_126', 'lag_127', 'lag_128', 'lag_129', 'lag_130', 'lag_131', 'lag_132', 'lag_133', 'lag_134', 'lag_135', 'lag_136', 'lag_137', 'lag_138', 'lag_139', 'lag_140', 'lag_141', 'lag_142', 'lag_143', 'lag_144', 'lag_145', 'lag_146', 'lag_147', 'lag_148', 'lag_149', 'lag_150', 'lag_151', 'lag_152', 'lag_153', 'lag_154', 'lag_155', 'lag_156', 'lag_157', 'lag_158', 'lag_159', 'lag_160', 'lag_161', 'lag_162', 'lag_163', 'lag_164', 'lag_165', 'lag_166

  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-26 18:23:22,231] Trial 0 finished with value: 940.9187200530015 and parameters: {'alpha': 1.0736148759814756, 'l1_ratio': 0.8481016486719523, 'fit_intercept': True}. Best is trial 0 with value: 940.9187200530015.
[I 2026-03-26 18:24:32,004] Trial 1 finished with value: 941.0493840319126 and parameters: {'alpha': 0.030933884147840207, 'l1_ratio': 0.655133819191949, 'fit_intercept': True}. Best is trial 0 with value: 940.9187200530015.
[I 2026-03-26 18:24:57,106] Trial 2 finished with value: 1367.18558621023 and parameters: {'alpha': 0.08199046638620358, 'l1_ratio': 0.4634924264996797, 'fit_intercept': False}. Best is trial 0 with value: 940.9187200530015.
[I 2026-03-26 18:25:05,437] Trial 3 finished with value: 943.7571151158211 and parameters: {'alpha': 0.772628468471332, 'l1_ratio': 0.004184025771115429, 'fit_intercept': True}. Best is trial 0 with value: 940.9187200530015.
[I 2026-03-26 18:28:02,675] Trial 4 finished with value: 941.4215719976157 and parameters: {'alpha': 

...

RF empirical-Bayes threshold (raw importance): 0.00461566
Number of selected features: 41
All features:
['unique_id', 'ds', 'y', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'precipitation', 'direct_radiation', 'lag_96', 'lag_97', 'lag_98', 'lag_99', 'lag_100', 'lag_101', 'lag_102', 'lag_103', 'lag_104', 'lag_105', 'lag_106', 'lag_107', 'lag_108', 'lag_109', 'lag_110', 'lag_111', 'lag_112', 'lag_113', 'lag_114', 'lag_115', 'lag_116', 'lag_117', 'lag_118', 'lag_119', 'lag_120', 'lag_121', 'lag_122', 'lag_123', 'lag_124', 'lag_125', 'lag_126', 'lag_127', 'lag_128', 'lag_129', 'lag_130', 'lag_131', 'lag_132', 'lag_133', 'lag_134', 'lag_135', 'lag_136', 'lag_137', 'lag_138', 'lag_139', 'lag_140', 'lag_141', 'lag_142', 'lag_143', 'lag_144', 'lag_145', 'lag_146', 'lag_147', 'lag_148', 'lag_149', 'lag_150', 'lag_151', 'lag_152', 'lag_153', 'lag_154', 'lag_155', 'lag_156', 'lag_157', 'lag_158', 'lag_159', 'lag_160', 'lag_161', 'lag_162', 'lag_163', 'lag_164', 'lag_165', 'lag_166

  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-03-27 03:29:36,839] Trial 0 finished with value: 412.15593807739134 and parameters: {'alpha': 1.3326203010793938, 'l1_ratio': 0.6962071615504555, 'fit_intercept': True}. Best is trial 0 with value: 412.15593807739134.
[I 2026-03-27 03:29:37,988] Trial 1 finished with value: 1318.4602861021297 and parameters: {'alpha': 0.2500096736775709, 'l1_ratio': 0.010030152465882125, 'fit_intercept': False}. Best is trial 0 with value: 412.15593807739134.
[I 2026-03-27 03:29:39,116] Trial 2 finished with value: 412.2716400846667 and parameters: {'alpha': 0.9068645898620686, 'l1_ratio': 0.44851298986222243, 'fit_intercept': True}. Best is trial 0 with value: 412.15593807739134.
[I 2026-03-27 03:29:40,188] Trial 3 finished with value: 414.6044003905466 and parameters: {'alpha': 2.8851414924371457, 'l1_ratio': 0.9570830727636499, 'fit_intercept': True}. Best is trial 0 with value: 412.15593807739134.
[I 2026-03-27 03:29:41,328] Trial 4 finished with value: 411.6195070783755 and parameters: {'a

c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.789e+05, tolerance: 2.233e+05
  model = cd_fast.enet_coordinate_descent(
c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.303e+05, tolerance: 2.299e+05
  model = cd_fast.enet_coordinate_descent(
c:\Users\CR58XM\AppData\Local\anaconda3\envs\sktime\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the

[I 2026-03-27 03:29:51,528] Trial 10 finished with value: 505.29781688642896 and parameters: {'alpha': 0.013217052189796085, 'l1_ratio': 0.9901986440667399, 'fit_intercept': True}. Best is trial 4 with value: 411.6195070783755.
[I 2026-03-27 03:29:52,628] Trial 11 finished with value: 415.1083692082797 and parameters: {'alpha': 8.478912328908349, 'l1_ratio': 0.6551122107110294, 'fit_intercept': True}. Best is trial 4 with value: 411.6195070783755.
[I 2026-03-27 03:29:53,756] Trial 12 finished with value: 412.4137660703558 and parameters: {'alpha': 7.187060043367892, 'l1_ratio': 0.764400148583338, 'fit_intercept': True}. Best is trial 4 with value: 411.6195070783755.
[I 2026-03-27 03:29:54,872] Trial 13 finished with value: 412.3807301799633 and parameters: {'alpha': 1.5717152894682285, 'l1_ratio': 0.46047425423748856, 'fit_intercept': True}. Best is trial 4 with value: 411.6195070783755.
[I 2026-03-27 03:29:56,038] Trial 14 finished with value: 416.5885769842783 and parameters: {'alpha

# end 

it takes around 3 hours